# CLV Thesis — Kaggle GPU Runner

**Purpose:** Run the deep-learning training sweep (LSTM + Transformer models) on Kaggle's free T4 GPU without the 8–14 h local runtime.

**Kaggle basics you need to know before running this notebook:**

| Concept | What it means |
|---------|---------------|
| `/kaggle/input/<slug>/` | Where your uploaded datasets are mounted — **read-only**. You cannot write here. |
| `/kaggle/working/` | The writable scratch space for this session. Outputs saved here survive when you download them. |
| Session limit | T4 GPU notebooks get **12 hours** of compute. After that the kernel dies. |
| Persistence | Code edits in the notebook **are saved** to your Kaggle account. Output files in `/kaggle/working/` must be explicitly downloaded or they disappear at session end. |
| Internet access | Off by default — turn it on in **Notebook settings → Internet** if you need `pip install` to reach PyPI. |

---

## Before you start: one-time Kaggle setup

### 1 · Upload your raw datasets

Each dataset must be uploaded to Kaggle as a **Dataset** so it is mounted at a stable path.  
Go to [kaggle.com/datasets/new](https://www.kaggle.com/datasets/new) for each one.

**Important:** Kaggle requires dataset titles to be at least 7 characters and derives the URL slug directly from the title. Type the slug value exactly as the title — all lowercase, no spaces.

| Files to upload | **Title to use (becomes the slug)** | Mounted at |
|----------------|--------------------------------------|------------|
| `CDNOW_sample.txt` | `cdnow-dataset` | `/kaggle/input/cdnow-dataset/` |
| `online_retail_II.xlsx` | `uci-retail` | `/kaggle/input/uci-retail/` |
| `ta_feng_all_months_merged.csv` | `tafeng-dataset` | `/kaggle/input/tafeng-dataset/` |
| `transactions.csv`, `hh_demographic.csv`, `campaign_table.csv`, `coupon_redempt.csv` | `dunnhumby` | `/kaggle/input/dunnhumby/` |

> The code in `src/utils/config.py` automatically maps these slugs back to the short  
> internal names (`cdnow`, `uci`, `tafeng`, `dunnhumby`) that your YAML configs use.  
> You do not need to rename anything in the repo.

### 2 · Add your datasets to this notebook

Open this notebook on Kaggle → right-hand panel → **"+ Add Data"** → search for each dataset you just uploaded → click **Add**.  
After adding, each dataset appears at its `/kaggle/input/<slug>/` path.

### 3 · Repository code is pulled from GitHub at runtime

Cell 1 runs `git clone https://github.com/OttoPrins/thesis-code-final.git` into  
`/kaggle/working/thesis-code/` every time you run it. There is **no `thesis-code-repo`  
Kaggle dataset to attach** — local edits + `git push` are picked up by the next session  
automatically. To pin a specific commit instead of `main`, set `os.environ["THESIS_REF"]`  
to a branch / tag / SHA before running Cell 1.

### 4 · Enable GPU

In the Kaggle notebook editor: **Settings (gear icon) → Accelerator → GPU T4 x2** (or just T4).  
Without this the notebook runs on CPU and will be much slower.

### 5 · Enable Internet access

**Settings → Internet → On** — needed so Cell 1 can run `pip install`.


---
## Cell 1 — Environment Setup

This cell does three things:
1. Installs the small set of packages that Kaggle does **not** pre-install.
2. Sets `KAGGLE_ENV=1` in the process environment so that every subsequent call to  
   `train.py` — including those launched as subprocesses by `run_seeds.py` — automatically  
   redirects paths to `/kaggle/input/` and `/kaggle/working/results/`.
3. Verifies that a GPU is visible to PyTorch.

**Run this cell first, every time you open the notebook.**

In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

# ── 1. Install missing packages (skipped if already present) ─────────────────
# We do NOT touch numpy here. Kaggle's base image ships NumPy 2.x and ~15
# preinstalled packages (shap, jax, cupy, opencv, pytensor, ...) require >=2.0.
# The codebase was audited 2026-05-13 and is NumPy 2.x compatible.
def _need_install(pkg_name):
    try:
        __import__(pkg_name)
        return False
    except ImportError:
        return True

to_install = []
if _need_install("lifetimes"):  to_install.append("lifetimes>=0.11.3")
if _need_install("openpyxl"):   to_install.append("openpyxl>=3.1.0")

if to_install:
    print(f"Installing: {to_install}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + to_install, check=True)
    print("Done.\n")
else:
    print("All packages already installed.\n")

# ── 2. Set Kaggle environment flag ────────────────────────────────────────────
os.environ["KAGGLE_ENV"] = "1"
print("KAGGLE_ENV=1 set.\n")

# ── 3. Clone (or refresh) the repo from GitHub ────────────────────────────────
# Internet must be ON: Settings → Internet → On. Pin a specific commit by
# setting THESIS_REF before running this cell, e.g. os.environ["THESIS_REF"]="<sha>".
REPO_URL  = "https://github.com/OttoPrins/thesis-code-final.git"
REPO_PATH = Path("/kaggle/working/thesis-code")
REPO_REF  = os.environ.get("THESIS_REF", "main")

if REPO_PATH.exists():
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "--all", "--tags", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", REPO_REF, "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{REPO_REF}", "--quiet"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_PATH)], check=True)

os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))
sha = subprocess.check_output(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--short", "HEAD"]
).decode().strip()
print(f"Repo : {REPO_PATH}  @ {sha}  (ref={REPO_REF})")


# ── 3.5. Ensure raw datasets are accessible ────────────────────────────────────
# CDNOW comes directly from the git repo (224 KB — no download needed).
# UCI has a UCI ML Repository fallback URL if the Kaggle dataset upload was skipped.
# TaFeng and Dunnhumby come from Kaggle (pre-mounted symlink or kaggle CLI download).
DATA_ROOT = Path("/kaggle/working/input")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── CDNOW: copy from git repo ─────────────────────────────────────────────────
cdnow_dst = DATA_ROOT / "cdnow-dataset"
cdnow_file = cdnow_dst / "CDNOW_sample.txt"
if cdnow_file.exists():
    print("  cdnow-dataset: already present.")
else:
    cdnow_dst.mkdir(exist_ok=True)
    shutil.copy(REPO_PATH / "data" / "raw" / "CDNOW_sample.txt", cdnow_file)
    print("  cdnow-dataset: copied from git repo.")

# ── UCI: Kaggle download with UCI ML Repo fallback ────────────────────────────
uci_dst = DATA_ROOT / "uci-retail"
if uci_dst.exists():
    print("  uci-retail: already present.")
elif Path("/kaggle/input/uci-retail").exists():
    uci_dst.symlink_to(Path("/kaggle/input/uci-retail"))
    print("  uci-retail: symlinked from /kaggle/input/.")
else:
    print("  uci-retail: not mounted — trying kaggle download ...")
    try:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", "ottoprins/uci-retail",
             "-p", str(uci_dst), "--unzip"],
            check=True, capture_output=True, timeout=300,
        )
        print("  uci-retail: kaggle download complete.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print(f"  uci-retail: kaggle download failed ({type(e).__name__}) — fetching from UCI ML Repo ...")
        uci_dst.mkdir(exist_ok=True)
        subprocess.run(["wget", "-q", "-O", "/tmp/uci.zip",
            "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"],
            check=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/uci.zip", "-d", str(uci_dst)],
            check=True)
        # UCI zip may extract with a slightly different name — normalise it
        for f in sorted(uci_dst.iterdir()):
            if f.suffix in (".xlsx", ".csv") and "retail" in f.name.lower():
                target_name = uci_dst / "online_retail_II.xlsx"
                if not target_name.exists():
                    f.rename(target_name)
                break
        print("  uci-retail: UCI fallback download complete.")

# ── TaFeng and Dunnhumby: standard Kaggle download / symlink ─────────────────
for slug, api_ref in [("tafeng-dataset", "ottoprins/tafeng-dataset"),
                       ("dunnhumby",      "ottoprins/dunnhumby")]:
    target  = DATA_ROOT / slug
    mounted = Path(f"/kaggle/input/{slug}")
    if target.exists():
        print(f"  {slug}: already present.")
    elif mounted.exists():
        target.symlink_to(mounted)
        print(f"  {slug}: symlinked from /kaggle/input/.")
    else:
        print(f"  {slug}: not mounted — downloading ...")
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", api_ref,
             "-p", str(target), "--unzip"],
            check=True,
        )
        print(f"  {slug}: download complete.")

# ── Confirm what's on disk ────────────────────────────────────────────────────
print("\n── Data directory contents ──")
for slug in ["cdnow-dataset", "uci-retail", "tafeng-dataset", "dunnhumby"]:
    d = DATA_ROOT / slug
    if d.exists():
        files = sorted(f.name for f in d.iterdir() if not f.name.startswith("."))[:6]
        print(f"  {slug}: {files}")
    else:
        print(f"  {slug}: [MISSING]")

os.environ["KAGGLE_DATA_ROOT"] = str(DATA_ROOT)
print(f"\nKAGGLE_DATA_ROOT={DATA_ROOT}  — all datasets ready.\n")

# ── 4. Verify GPU (fail fast if incompatible) ─────────────────────────────────
import torch
print(f"\ntorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_capability(0)
    print(f"GPU  : {name}  (sm_{cap[0]}{cap[1]})")
    print(f"VRAM : {vram:.1f} GB")
    if cap < (7, 0):
        raise RuntimeError(
            f"\n  {name} has CUDA capability sm_{cap[0]}{cap[1]} "
            f"(PyTorch {torch.__version__} requires sm_70+).\n"
            "  Fix: Settings -> Accelerator -> GPU T4 x1 or T4 x2\n"
            "       then restart the kernel and re-run Cell 1."
        )
    print(f"GPU capability OK (sm_{cap[0]}{cap[1]} >= 7.0).")
else:
    raise RuntimeError(
        "No GPU found. Enable one: Settings -> Accelerator -> GPU T4 x1 (or T4 x2)."
    )

# ── 5. Output directory ───────────────────────────────────────────────────────
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
print("\n/kaggle/working/results/ — ready.")


---
## Cell 2 — Data Path Verification

Before training anything, confirm that Kaggle has mounted your datasets correctly  
and that the data pipeline can actually read the files.

**What to check:** Every dataset you added in the setup steps should show `✓` below.  
If you see `✗ NOT MOUNTED`, go back to the Kaggle notebook editor → **"+ Add Data"** and add the missing dataset, then re-run this cell.

The pipeline validation (`validate_pipelines.py`) does a lightweight end-to-end check:  
it loads the raw file, runs the weekly aggregation, and verifies tensor shapes — without  
starting any training. It takes ~10 seconds per dataset.

In [ ]:
import os
from pathlib import Path

# ── Check dataset mounts ──────────────────────────────────────────────────────
# After Cell 1, KAGGLE_DATA_ROOT=/kaggle/working/input — check there.
# On a fresh interactive session without Cell 1 run, falls back to /kaggle/input
# (where Dunnhumby is still pre-mounted from the UI attachment).
data_root = os.environ.get("KAGGLE_DATA_ROOT", "/kaggle/input")

EXPECTED_FILES = {
    "cdnow-dataset" : ["CDNOW_sample.txt"],
    "uci-retail"    : ["online_retail_II.xlsx"],
    "tafeng-dataset": ["ta_feng_all_months_merged.csv"],
    "dunnhumby"     : ["transaction_data.csv", "hh_demographic.csv"],
}

all_ok = True
for slug, expected in EXPECTED_FILES.items():
    p = Path(data_root) / slug
    label = f"{data_root}/{slug}/"
    if not p.exists():
        print(f"✗  {label}  — NOT FOUND  (run Cell 1 first)")
        all_ok = False
        continue
    actual = sorted(f.name for f in p.iterdir())
    missing = [f for f in expected if f not in actual]
    if missing:
        print(f"⚠  {label}  — found but missing: {missing}")
        print(f"   Files present: {actual[:10]}")
        all_ok = False
    else:
        print(f"✓  {label}  — {len(actual)} file(s): {actual[:5]}")

print()
if not all_ok:
    print("Fix the missing datasets above (run Cell 1) before running training cells.")
else:
    print("All datasets ready.")

# ── Quick pipeline validation (CDNOW only — takes ~10 s) ──────────────────────
# This runs the data pipeline end-to-end and checks tensor shapes.
# It does NOT train a model.
print("\nRunning pipeline validation for CDNOW...")
!python validate_pipelines.py --dataset cdnow

# Uncomment to validate other datasets:
# !python validate_pipelines.py --dataset uci
# !python validate_pipelines.py --dataset tafeng
# !python validate_pipelines.py --dataset dunnhumby


---
## Cell 3 — Training: CDNOW + UCI  (~2.5–4 h on T4)

This cell runs the full 3-seed sweep for CDNOW and UCI using `run_seeds.py`.  
`run_seeds.py` is your existing multi-seed orchestrator — it calls `train.py` as a  
subprocess for each `(config, seed)` pair and skips runs whose output already exists.

**Why CDNOW + UCI first?**  
They are the smallest datasets. If you run out of time before reaching TaFeng or Dunnhumby  
you will still have meaningful results to download. Cell 5 archives whatever exists.

**Expected output:**  
For each run you will see per-epoch loss lines followed by holdout metrics, then:
```
Metrics saved: /kaggle/working/results/tables/lstm_base_cdnow_final_seed42_sample_metrics.json
```
All checkpoints land in `/kaggle/working/results/checkpoints/`.

**`KAGGLE_ENV=1` is already set** from Cell 1 — you do not need `--kaggle` here  
because `run_seeds.py` launches subprocesses that inherit the environment variable.

In [ ]:
# ── CDNOW + UCI: 6 configs × 3 seeds × 1 mode = 18 runs ─────────────────────
# --skip_existing means if you re-run this cell, already-completed runs are skipped.
!python run_seeds.py \
    --configs lstm_base_cdnow_v2 lstm_joint_cdnow_v2 transformer_joint_cdnow_v2 \
              lstm_base_uci_v2   lstm_joint_uci_v2   transformer_joint_uci_v2 \
    --seeds 42 7 123 \
    --modes sample \
    --skip_existing


---
## Cell 4 — Training: Dunnhumby  (~1.5–3 h on T4)

Run this after Cell 3 completes if you still have session time remaining.  
Check the session timer in the Kaggle editor (top-right) — you need at least ~2 h free.

In [ ]:
# ── Dunnhumby: 3 configs × 3 seeds = 9 runs ─────────────────────────────────
!python run_seeds.py \
    --configs lstm_base_dunnhumby_v2 lstm_joint_dunnhumby_v2 transformer_joint_dunnhumby_v2 \
    --seeds 42 7 123 \
    --modes sample \
    --skip_existing


---
## Cell 5 — Training: TaFeng  (~4–7 h on T4, run LAST)

TaFeng has ~32,000 customers — the largest dataset by far. Run this last because  
it is most likely to hit the 12-hour limit.

**If you expect to run out of time:** reduce `--seeds 42` (single seed) and run  
the 3-seed sweep in a second Kaggle session using `--skip_existing`.

In [ ]:
# ── TaFeng: 3 configs × 3 seeds = 9 runs ─────────────────────────────────────
# If you hit OOM errors, add --batch_size_override 64 once that flag is added,
# or manually edit the YAML batch_size to 64 and re-run.
!python run_seeds.py \
    --configs lstm_base_tafeng_v2 lstm_joint_tafeng_v2 transformer_joint_tafeng_v2 \
    --seeds 42 7 123 \
    --modes sample \
    --skip_existing


---
## Cell 6 — Single targeted run  (debug / one-off)

Use this cell to run a single config directly — useful for debugging, checking that  
paths are correct, or running a quick smoke test before committing to the full sweep.

The `--kaggle` flag here is redundant (we already set `KAGGLE_ENV=1` in Cell 1), but  
it is included as an explicit reminder that the override is active.

**Smoke test** (3 epochs, 2 scenarios — completes in ~1 minute):

In [ ]:
# ── Smoke test — single config, 3 epochs, 2 MC scenarios (~1 min) ─────────────
!python train.py \
    --config experiments/configs/lstm_base_cdnow.yaml \
    --kaggle \
    --seed_override 42 \
    --max_epochs 3 \
    --n_scenarios 2

# ── Full single run (one config, one seed, full 100 epochs) ───────────────────
# Uncomment to use:
# !python train.py \
#     --config experiments/configs/transformer_joint_cdnow.yaml \
#     --kaggle \
#     --seed_override 42

# ── Run with a custom data root ───────────────────────────────────────────────
# If all your raw files are in a single Kaggle dataset (e.g. slug = "thesis-data"),
# use --kaggle-data-root to point to that directory instead of /kaggle/input:
# !python train.py \
#     --config experiments/configs/lstm_base_cdnow.yaml \
#     --kaggle \
#     --kaggle-data-root /kaggle/input/thesis-data \
#     --seed_override 42

---
## Cells v3.1–v3.6 — v3 + HPO pipeline (scheduled sampling + spend fix)

The v2 cells above are the **pure-teacher-forced Stage-1 replication baseline** — keep them.
These v3 cells produce the *improved* models:

| Step | Cell | What it does |
|------|------|--------------|
| v3.2 | install | ensure `optuna` is present |
| v3.3 | **HPO** | tune scheduled-sampling dose + loss weighting + capacity for all 12 v3 configs → `*_v3_hpo.yaml` |
| v3.4 | final | 3-seed (42/7/123) evaluation with the HPO-selected configs |
| v3.5 | ensemble | seed-ensemble the 3 seeds (residual-variance reducer) |
| v3.6 | tables | rebuild `comparison_all.csv` / `comparison_seeds.csv` (v2 vs v3_hpo vs ensemble + expected diagnostic) |

**Why:** the v2 autoregressive rollout collapses on sparse data (CDNOW frequency bias
−31% ± 21%); scheduled sampling fixes it but its dose must be tuned (an untuned smoke
flipped bias to +45%). Spend R² was negative on 3/4 datasets because Kendall silenced
the spend task — `spend_logvar_max` (in the v3 configs) prevents that.

**Heavy.** Run `v3.3` first; it is by far the longest. Tune `N_TRIALS` / `MAX_EPOCHS`
in that cell to your GPU budget. Every study is resumable (sqlite), so a 12 h timeout
mid-sweep is fine — just re-run the cell next session. Run **Cell 7** (archive) before
the session ends regardless of progress.

In [ ]:
# ── v3.2 — Ensure optuna is available (idempotent) ───────────────────────────
import sys, subprocess
try:
    import optuna  # noqa: F401
    print(f"optuna {optuna.__version__} already installed")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "optuna>=3.0.0"], check=True)
    import optuna  # noqa: F401
    print(f"optuna {optuna.__version__} installed")

In [ ]:
# ── v3.3 — Phase 4: HPO over all 12 v3 configs ───────────────────────────────
# Tunes scheduled-sampling dose + Kendall/loss weighting + capacity per config
# on the balanced objective. Each study writes experiments/configs/<name>_hpo.yaml.
# HEAVY. Total ≈ 12 × N_TRIALS × MAX_EPOCHS model-fits. Lower N_TRIALS / MAX_EPOCHS
# if GPU-time-constrained; the sqlite store makes every study RESUMABLE across
# sessions (re-run this cell after a timeout to continue).
import subprocess, sys

V3_CONFIGS = [
    "lstm_base_cdnow_v3", "lstm_joint_cdnow_v3", "transformer_joint_cdnow_v3",
    "lstm_base_uci_v3", "lstm_joint_uci_v3", "transformer_joint_uci_v3",
    "lstm_base_tafeng_v3", "lstm_joint_tafeng_v3", "transformer_joint_tafeng_v3",
    "lstm_base_dunnhumby_v3", "lstm_joint_dunnhumby_v3", "transformer_joint_dunnhumby_v3",
]
N_TRIALS   = 40    # ← lower (e.g. 20) if GPU-budget constrained
MAX_EPOCHS = 120   # trial cap; the FINAL run (next cell) uses the full 300
N_SCEN     = 50    # trial inference scenarios; final run uses 200

for name in V3_CONFIGS:
    print(f"\n========== HPO {name} ==========", flush=True)
    subprocess.run([sys.executable, "tune.py",
        "--config", f"experiments/configs/{name}.yaml",
        "--kaggle",
        "--n-trials", str(N_TRIALS),
        "--max-epochs", str(MAX_EPOCHS),
        "--n-scenarios", str(N_SCEN),
        "--storage", f"sqlite:////kaggle/working/hpo_{name}.db",
        "--study-name", f"hpo_{name}"], check=False)
print("\nHPO complete. Winners: experiments/configs/*_v3_hpo.yaml")

In [ ]:
# ── v3.4 — Phase 2: final 3-seed evaluation with the HPO-selected configs ─────
# Uses each *_v3_hpo.yaml's full epoch budget (300). KAGGLE_ENV=1 (Cell 1) makes
# run_seeds.py propagate Kaggle path overrides to its train.py subprocesses.
# --skip_existing resumes if a prior session timed out mid-sweep.
import subprocess, sys

HPO_CONFIGS = [
    "lstm_base_cdnow_v3_hpo", "lstm_joint_cdnow_v3_hpo", "transformer_joint_cdnow_v3_hpo",
    "lstm_base_uci_v3_hpo", "lstm_joint_uci_v3_hpo", "transformer_joint_uci_v3_hpo",
    "lstm_base_tafeng_v3_hpo", "lstm_joint_tafeng_v3_hpo", "transformer_joint_tafeng_v3_hpo",
    "lstm_base_dunnhumby_v3_hpo", "lstm_joint_dunnhumby_v3_hpo", "transformer_joint_dunnhumby_v3_hpo",
]
subprocess.run([sys.executable, "run_seeds.py",
    "--configs", *HPO_CONFIGS,
    "--seeds", "42", "7", "123",
    "--modes", "sample",
    "--skip_existing"], check=False)
print("\nFinal multi-seed metrics + checkpoints under /kaggle/working/results/")

In [ ]:
# ── v3.5 — Seed-ensemble the 3 seeds of each HPO config (variance reducer) ────
# Averages the per-week predictions of seeds 42/7/123 (sample mode) and rescoring
# with the standard metric stack. Reduces residual cross-seed variance.
import subprocess, sys

HPO_CONFIGS = [
    "lstm_base_cdnow_v3_hpo", "lstm_joint_cdnow_v3_hpo", "transformer_joint_cdnow_v3_hpo",
    "lstm_base_uci_v3_hpo", "lstm_joint_uci_v3_hpo", "transformer_joint_uci_v3_hpo",
    "lstm_base_tafeng_v3_hpo", "lstm_joint_tafeng_v3_hpo", "transformer_joint_tafeng_v3_hpo",
    "lstm_base_dunnhumby_v3_hpo", "lstm_joint_dunnhumby_v3_hpo", "transformer_joint_dunnhumby_v3_hpo",
]
for name in HPO_CONFIGS:
    cfg = f"experiments/configs/{name}.yaml"
    print(f"\n===== seed-ensemble {name} =====", flush=True)
    subprocess.run([sys.executable, "-m", "src.evaluation.seed_ensemble",
        "--config", cfg,
        "--seeds", "42", "7", "123", "--mode", "sample",
        "--checkpoint-dir", "/kaggle/working/results/checkpoints",
        "--results-dir", "/kaggle/working/results",
        "--kaggle",
        "--fit-temperature", "--fit-aggregate-calibration"], check=False)
print("\nSeed-ensemble metrics written under /kaggle/working/results/tables/")

In [ ]:
# ── v3.6 — Rebuild comparison tables (v2 baseline vs v3_hpo vs ensemble) ──────
# --include_exploratory: include the new non-manifest v3/hpo/ensemble runs
# --include_expected:    surface the expected-mode diagnostic column
# --seeds: also write comparison_seeds.csv (mean ± std across 42/7/123)
!python -m src.evaluation.compare \
    --results_dir /kaggle/working/results \
    --include_expected --include_exploratory --seeds --latex
print("Refreshed: /kaggle/working/results/tables/comparison_all.csv & comparison_seeds.csv")
print("Now run Cell 7 to archive /kaggle/working/results for download.")

---
## Cell 7 — Archive results for download

**Always run this cell before the session ends**, even if training is still running  
in other cells — it archives whatever has been saved so far.

After running:
1. In the Kaggle notebook editor, click **"Save & Run All"** (or just save).
2. Go to the **Output** tab (right panel or bottom of the page).
3. Find `results_archive.zip` and click **Download**.

The archive contains:
- `tables/` — `*_metrics.json` and `*_history.json` for every completed run
- `checkpoints/` — `.pt` model weights
- `plots/` — any figures generated by the evaluation scripts

> **Note:** Kaggle also shows individual output files under the Output tab.  
> The zip just makes bulk downloading easier.

In [ ]:
import shutil, os
from pathlib import Path

results_dir  = Path("/kaggle/working/results")
archive_stem = "/kaggle/working/results_archive"   # .zip will be appended automatically

if not results_dir.exists() or not any(results_dir.rglob("*")):
    print("No results found yet — run at least one training cell first.")
else:
    # Report what we have before archiving
    metrics_files = sorted(results_dir.rglob("*_metrics.json"))
    ckpt_files    = sorted(results_dir.rglob("*.pt"))
    print(f"Metrics files  : {len(metrics_files)}")
    print(f"Checkpoints    : {len(ckpt_files)}")
    for f in metrics_files:
        print(f"  {f.name}")

    # Build zip
    shutil.make_archive(archive_stem, "zip", results_dir)
    archive_path = Path(archive_stem + ".zip")
    size_mb = archive_path.stat().st_size / (1024 ** 2)
    print(f"\nArchive created : {archive_path}  ({size_mb:.1f} MB)")
    print("Download via    : Kaggle notebook → Output tab → results_archive.zip")

---
## Troubleshooting

### Notebook on Kaggle still shows old cells after `bash push_to_kaggle.sh`
The Kaggle web editor caches the notebook revision in your browser tab. After  
pushing, **hard-refresh the tab** (⌘⇧R on macOS, Ctrl-Shift-R on Linux/Windows)  
before clicking Run All. Otherwise you re-run the previously cached version.

### `git clone` fails in Cell 1
Internet is off. Open **Settings → Internet → On**, then re-run Cell 1.


### `RuntimeError: … sm_60 … requires sm_70+`
Kaggle allocated a P100 (Pascal) GPU, which PyTorch 2.x no longer supports.
Fix: **Settings → Accelerator → GPU T4 x1** (or GPU T4 x2), then restart the
kernel and re-run Cell 1. T4 (sm_75) is well-supported.

### `FileNotFoundError: CDNOW raw file not found in /kaggle/input/cdnow-dataset`
The Kaggle dataset slug doesn't match what the code expects. The slug mapping is defined in
`src/utils/config.py` → `_KAGGLE_SLUG_MAP`. Expected slugs: `cdnow-dataset`, `uci-retail`,
`tafeng-dataset`, `dunnhumby`. If you used a different title when uploading, either
re-create the dataset with the correct title, or override the path in Cell 1:
```python
import os
os.environ["KAGGLE_DATA_ROOT"] = "/kaggle/input/my-actual-slug"
os.environ["KAGGLE_ENV"] = "1"
```

### `CUDA out of memory` on TaFeng
TaFeng has ~32 k customers. Reduce the batch size by copying the config to `/kaggle/working/`,
editing `training.batch_size: 64`, and passing the new path:
```bash
!cp experiments/configs/lstm_base_tafeng.yaml /kaggle/working/
# edit the file, then:
!python train.py --config /kaggle/working/lstm_base_tafeng.yaml --kaggle --seed_override 42
```
Note: config files are in the read-only repo dataset, so they must be copied to `/kaggle/working/` before editing.

### `ModuleNotFoundError: No module named 'src'`
`os.chdir(REPO_PATH)` in Cell 1 must point at the directory that contains `src/` and `train.py`.  
Double-check `REPO_PATH` and that the repository files were uploaded correctly.

### Session times out mid-sweep
`run_seeds.py --skip_existing` resumes from where it left off — just re-open the notebook,  
run Cell 1 (to set `KAGGLE_ENV=1` again), then re-run the relevant training cell.  
Completed runs are detected by their existing `*_metrics.json` files and skipped automatically.

### `rpy2` / `cmdstanpy` import errors
These are **not needed** for DL training. If you accidentally call `run_benchmarks.py`  
on Kaggle it will fail — that script requires R and Stan. Only run it locally.  
The cells in this notebook never call `run_benchmarks.py`.

### Probabilistic benchmarks (Pareto/NBD, Pareto/GGG, GPPM)
These cannot run on Kaggle (missing R / Stan). Run them locally with your full environment  
and merge the resulting `*_metrics.json` files into the same `results/tables/` directory  
before generating comparison tables.

